In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

def l2norm(x, dim=-1, eps=1e-6):
    return x * torch.rsqrt((x * x).sum(dim=dim, keepdim=True) + eps)


class GatedDeltaNet(nn.Module):
    def __init__(
        self, d_in, d_out, dropout, num_heads, qkv_bias=False
    ):
        super().__init__()
        assert d_out % num_heads == 0

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        #  Gates for delta rule and output gating
        self.W_gate = nn.Linear(d_in, d_out, bias=False)
        self.W_beta = nn.Linear(d_in, d_out, bias=False) # Write Strength

        # The decay gate alpha corresponds to
        # A_log + W_alpha(x) + dt_bias   
        # controls how much of the past memory $S$ should be "forgotten". 
        # It is data-dependent, meaning the model dynamically decides to wipe its memory if it sees a sudden change in context.
        self.W_alpha = nn.Linear(d_in, num_heads, bias=False) # or like  # W_alpha = nn.Linear(d_in, num_heads, bias=True)
        self.dt_bias = nn.Parameter(torch.ones(num_heads))
        A_init = torch.empty(num_heads).uniform_(0, 16)
        self.A_log = nn.Parameter(torch.log(A_init))

        self.norm = nn.RMSNorm(self.head_dim, eps=1e-6)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        b, num_tokens, _ = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # Delta Rule
        beta = torch.sigmoid(self.W_beta(x))
        alpha_log = -self.A_log.exp().view(1,1,-1) * F.softplus(self.W_alpha(x) + self.dt_bias)
        alpha = alpha_log.exp()
        gate = self.W_gate(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        beta = beta.view(b, num_tokens, self.num_heads, self.head_dim)
        gate = gate.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)
        beta = beta.transpose(1,2)
        

        # L2normalization
        queries = l2norm(queries, dim=-1)/ (self.head_dim **0.5)
        keys = l2norm(keys, dim=-1)

        # fixed-size memory matrix
        S = x.new_zeros(b, self.num_heads, self.head_dim, self.head_dim)

        outs = []
        ### Gated delta rule update
        for t in range(num_tokens):
            k_t = keys[:, :, t]
            q_t = queries[:, :, t]
            v_t = values[:, :, t]
            b_t = beta[:, :, t]
            a_t = alpha[:,t].unsqueeze(-1).unsqueeze(-1)
            S = S * a_t # how much information to forget
            kv_mem = (S * k_t.unsqueeze(-1)).sum(dim=-2) # predict from memory: Given the current Key ($k_t$), what does my memory already know about this topic?
            delta = (v_t - kv_mem) * b_t # subtracts its prediction from the actual value to find the error or novelty, multiplied by writing strengths 
            S = S + k_t.unsqueeze(-1) * delta.unsqueeze(-2) # Memory update: outer product of the Key and the Delta
            y_t = (S * q_t.unsqueeze(-1)).sum(dim=-2) # final output: Memory * Query
            outs.append(y_t) 
        
        context = torch.stack(outs, dim=2).transpose(1, 2).contiguous()
        context = context.view(b, num_tokens, self.num_heads, self.head_dim)

        context = self.norm(context)
        context = context * F.silu(gate)


        context = context.view(b, num_tokens, self.d_out)
        context = self.dropout(context)
        out = self.out_proj(context)
        return out


batch_size = 2
seq_len = 5
d_in = 16
d_out = 32
num_heads = 4
dropout = 0.1

# Instantiate model
gated_deltanet = GatedDeltaNet(
    d_in=d_in,
    d_out=d_out,
    dropout=dropout,
    num_heads=num_heads
)

x = torch.randn(batch_size, seq_len, d_in)
# print(gated_deltanet(x))